In [1]:
import torch
import wandb as wb
import yaml
import itertools
import time

from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from model import CMCDNet
from loss import LossFunction, BCEFocalLoss
from dataset import ChangeDetctionDataset
from transform import PairedTransform, ValTransform
from sampler import BucketBatchSampler
from utils.logging import log_images
from utils.eval_metrics import iou_score, precision_recall, f1_score

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

data = cfg["data"]
hyperparameter = cfg["training"]
aug = cfg["augmentations"]

In [ ]:
### Train Dataset
dataset = ChangeDetctionDataset(
    pre_img_path=data["train"]["pre_event"],
    post_img_path=data["train"]["post_event"],
    target_img_path=data["train"]["target"],
    patch_size=data["patch_size"],
    stride=data["stride"],
    index_path=data["index_path"],
    build_metadata=True,
    transform=PairedTransform(
        horizontal_flip_p=aug["horizontal_flip"]["probability"],
        vertical_flip_p=aug["vertical_flip"]["probability"]
    )
)

batch_sampler = BucketBatchSampler(
    dataset.patch_metadata,
    batch_size = hyperparameter["batch_size"],
    ratio=cfg["sampler"]["ratios"],
)

train_loader = DataLoader(
    dataset,
    batch_sampler=batch_sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,)

In [4]:
### Validaation Dataset
validation_dataset = ChangeDetctionDataset(
    pre_img_path=data["val"]["pre_event"],
    post_img_path=data["val"]["post_event"],
    target_img_path=data["val"]["target"],
    patch_size=data["patch_size"],
    stride=data["stride"],
    index_path=None,
    build_metadata=False,
    transform=ValTransform()
)

val_loader = DataLoader(
    validation_dataset,
    batch_size=hyperparameter["batch_size"],
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CMCDNet().to(device)
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")   
    model = torch.compile(model, mode="default") 
    scaler = GradScaler("cuda")
# criterion = BCEFocalLoss()
criterion = LossFunction()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hyperparameter["learning_rate"],
    weight_decay=hyperparameter["weight_decay"], 
)

size = sum(1 for m in dataset.patch_metadata if m["bucket"] != "discard")

steps_per_epoch = size // hyperparameter["batch_size"] ##15827.875
print(steps_per_epoch)
global_step = 0
# loss_tracker = {}

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Using GPU: NVIDIA RTX A5000


KeyboardInterrupt: 

In [6]:
from collections import Counter

bucket_counts = Counter(m["bucket"] for m in dataset.patch_metadata)
print(bucket_counts)
print("total:", sum(bucket_counts.values()))

Counter({'trivial': 37698, 'hard_negative': 21154, 'informative': 5674, 'discard': 4999})
total: 69525


In [8]:
hyperparameter["epochs"] = 1

In [ ]:
import os

for epoch in range(hyperparameter["epochs"]):
    pbar = tqdm(itertools.islice(train_loader, steps_per_epoch), total=steps_per_epoch, leave=False)
    for step_idx, (pre, post, target, _) in enumerate(pbar):
        pre = pre.to(device)
        post = post.to(device)
        target = target.to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(pre, post)
            loss = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += loss.item()
        global_step += 1

        current_step = step_idx + 1
        current_lr = optimizer.param_groups[0]['lr']
        

        pbar.set_description(f"Epoch {epoch+1}")
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{current_lr:.2e}"})

    print()
    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        iou_loss = 0.0
        total_tp, total_fp, total_fn = 0, 0, 0
        for pre, post, target, _ in val_loader:
            pre = pre.to(device)
            post = post.to(device)
            target = target.to(device)
            logits = model(pre, post)
            loss = criterion(logits, target)
            val_loss += loss.item()
            iou_loss += iou_score(logits, target)

            pred = (torch.sigmoid(logits) > cfg['training']["threshold"])
            target_mask = target.unsqueeze(1) if target.dim() == 3 else target
            target_mask = target_mask.bool()
            tp = (pred & target_mask).sum().item()
            fp = (pred & ~target_mask).sum().item()
            fn = (~pred & target_mask).sum().item()
            total_tp += tp
            total_fp += fp
            total_fn += fn
            intersection = (pred_mask & target_mask.bool()).float().sum((1, 2, 3))
            union = (pred_mask | target_mask.bool()).float().sum((1, 2, 3)).clamp_min(1)

        precision = total_tp / (total_tp + total_fp + 1e-3)
        recall = total_tp / (total_tp + total_fn + 1e-3)
        f1 = (2 * precision * recall) / (precision + recall + 1e-3)
        val_avg_loss = val_loss / len(val_loader)
        val_avg_iou = iou_loss / len(val_loader)
        train_epoch_loss = running / steps_per_epoch

        val_log = {
            "val/loss": val_avg_loss,
            "val/iou": val_avg_iou,
            "val/precision": precision,
            "val/recall": recall,
            "val/f1_score": f1,
            "val/learning_rate": current_lr
        }
        print(
            f"epoch {epoch + 1}/{hyperparameter['epochs']} | "
            f"train_loss {train_epoch_loss:.4f} | "
            f"val_loss {val_avg_loss:.4f} | "
            f"val_iou {val_avg_iou:.4f} | "
            f"val_f1 {f1:.4f} | "
        )


  0%|          | 0/1152 [00:00<?, ?it/s]

W0512 18:13:25.942000 319236 torch/_functorch/_aot_autograd/autograd_cache.py:1101] [0/3] AOTAutograd cache unable to serialize compiled graph: Please convert all Tensors to FakeTensors first or instantiate FakeTensorMode with 'allow_non_fake_inputs'. Found in aten._to_copy.default(tensor([...], device='cuda:0', size=(4,), dtype=torch.uint8), device=device(type='cpu'))



epoch 1/1 | train_loss 0.3251 | val_loss 0.2424 | val_iou 0.0524 | val_f1 0.6070 | 


In [11]:
# Example: Save model and optimizer states
torch.save({
'epoch': 10,
'model_state_dict': model.state_dict(),
'optimizer_state_dict': optimizer.state_dict(),
'loss': loss.item(),
}, "model_checkpoint.pth")

In [ ]:
import os

with wb.init(project="EO-SAR Change-Detection", name="cmcdnet-run-002", config=cfg) as run:
    best_val_iou = float("-inf")
    ckpt_dir = "checkpoints/v2"

    for epoch in range(hyperparameter["epochs"]):
        start = time.perf_counter()
        model.train()
        running = 0.0
        pbar = tqdm(itertools.islice(train_loader, steps_per_epoch), total=steps_per_epoch, leave=False)
        for step_idx, (pre, post, target, _) in enumerate(pbar):
            pre = pre.to(device)
            post = post.to(device)
            target = target.to(device)

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(pre, post)
                loss = criterion(logits, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running += loss.item()
            global_step += 1

            current_step = step_idx + 1
            current_lr = optimizer.param_groups[0]['lr']
            

            pbar.set_description(f"Epoch {epoch+1}")
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{current_lr:.2e}"})

            if global_step % cfg["metrics"]["log_steps"] == 0:
                iou = iou_score(logits, target)
                precision, recall = precision_recall(logits, target)
                f1 = f1_score(precision, recall)

                log = {
                    "train/loss": loss.item(),
                    "train/iou": iou,
                    "train/precision": precision,
                    "train/recall": recall,
                    "train/f1_score": f1,
                    "train/learning_rate": current_lr
                }

                run.log(log, step=global_step)

            if cfg["metrics"]["log_images"] and global_step % cfg["metrics"]["image_log_every"] == 0:
                pred = (torch.sigmoid(logits) > cfg['training']["threshold"]).squeeze(1)
                log_images(pre, post, pred, target, step=global_step, max_images=2)

        print()
        model.eval()
        with torch.no_grad():
            val_loss = 0.0
            iou_loss = 0.0
            total_tp, total_fp, total_fn = 0, 0, 0
            for pre, post, target, _ in val_loader:
                pre = pre.to(device)
                post = post.to(device)
                target = target.to(device)
                logits = model(pre, post)
                loss = criterion(logits, target)
                val_loss += loss.item()
                iou_loss += iou_score(logits, target)

                pred = (torch.sigmoid(logits) > cfg['training']["threshold"])
                target_mask = target.unsqueeze(1) if target.dim() == 3 else target
                target_mask = target_mask.bool()
                tp = (pred & target_mask).sum().item()
                fp = (pred & ~target_mask).sum().item()
                fn = (~pred & target_mask).sum().item()
                total_tp += tp
                total_fp += fp
                total_fn += fn

            precision = total_tp / (total_tp + total_fp + 1e-3)
            recall = total_tp / (total_tp + total_fn + 1e-3)
            f1 = (2 * precision * recall) / (precision + recall + 1e-3)
            val_avg_loss = val_loss / len(val_loader)
            val_avg_iou = iou_loss / len(val_loader)
            train_epoch_loss = running / steps_per_epoch

            val_log = {
                "val/loss": val_avg_loss,
                "val/iou": val_avg_iou,
                "val/precision": precision,
                "val/recall": recall,
                "val/f1_score": f1,
                "val/learning_rate": current_lr
            }

            run.log(val_log, step=epoch)
            if val_avg_iou > best_val_iou:
                best_val_iou = val_avg_iou
                ckpt_path = os.path.join(ckpt_dir, f"best_epoch{epoch+1}_iou{best_val_iou:.4f}.pth")
                torch.save({
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict() if 'scaler' in globals() else None,
                    "best_val_iou": best_val_iou,
                }, ckpt_path)
                print(f"Saved new best checkpoint: {ckpt_path}")

                artifact = wb.Artifact(
                    name="cmcdnet-best",
                    type="model",
                    metadata={"epoch": epoch + 1, "best_val_iou": float(best_val_iou)}
                )
                artifact.add_file(ckpt_path)
                run.log_artifact(artifact)
                artifact.wait() 

        epoch_time = time.perf_counter() - start
        print(
            f"epoch {epoch + 1}/{hyperparameter['epochs']} | "
            f"train_loss {train_epoch_loss:.4f} | "
            f"val_loss {val_avg_loss:.4f} | "
            f"val_iou {val_avg_iou:.4f} | "
            f"val_f1 {f1:.4f} | "
            f"time {epoch_time:.2f}s"
        )
        run.log({"train/per_epoch_sec": epoch_time}, step=epoch)
        run.log({"train/epoch_loss": train_epoch_loss}, step=steps_per_epoch)

In [7]:
import os


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CMCDNet().to(device)
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")   
    scaler = GradScaler("cuda")
# criterion = BCEFocalLoss()
criterion = LossFunction()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hyperparameter["learning_rate"],
    weight_decay=hyperparameter["weight_decay"], 
)

size = sum(1 for m in dataset.patch_metadata if m["bucket"] != "discard")
steps_per_epoch = size // hyperparameter["batch_size"] ##15827.875

checkpoint = torch.load("checkpoints/v2/best_epoch7_iou0.0531.pth", map_location=device)


state_dict = checkpoint.get("model_state_dict", checkpoint)
for prefix in ("_orig_mod.", "module."):
    if any(k.startswith(prefix) for k in state_dict.keys()):
        state_dict = {k.replace(prefix, "", 1): v for k, v in state_dict.items()}
        break
model.load_state_dict(state_dict, strict=True)


if device.type == "cuda":
    model = torch.compile(model, mode="default")

o_state_dict = checkpoint.get("optimizer_state_dict", checkpoint)
optimizer.load_state_dict(o_state_dict)

start_epoch = checkpoint["epoch"] + 1
if checkpoint.get("scaler_state_dict") and device.type == "cuda":
    scaler.load_state_dict(checkpoint["scaler_state_dict"])


ckpt_dir = "checkpoints/v2"
global_step = checkpoint.get("global_step", 0)
model.train()
for epoch in range(start_epoch, hyperparameter["epochs"]):
    running = 0.0
    pbar = tqdm(itertools.islice(train_loader, steps_per_epoch), total=steps_per_epoch, leave=False)
    for step_idx, (pre, post, target, _) in enumerate(pbar):
        pre = pre.to(device)
        post = post.to(device)
        target = target.to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(pre, post)
            loss = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += loss.item()
        global_step += 1

        current_step = step_idx + 1
        current_lr = optimizer.param_groups[0]['lr']
        

        pbar.set_description(f"Epoch {epoch+1}")
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{current_lr:.2e}"})

    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        iou_loss = 0.0
        total_tp, total_fp, total_fn = 0, 0, 0
        for pre, post, target, _ in val_loader:
            pre = pre.to(device)
            post = post.to(device)
            target = target.to(device)
            logits = model(pre, post)
            loss = criterion(logits, target)
            val_loss += loss.item()
            iou_loss += iou_score(logits, target)

            pred = (torch.sigmoid(logits) > cfg['training']["threshold"])
            target_mask = target.unsqueeze(1) if target.dim() == 3 else target
            target_mask = target_mask.bool()
            tp = (pred & target_mask).sum().item()
            fp = (pred & ~target_mask).sum().item()
            fn = (~pred & target_mask).sum().item()
            total_tp += tp
            total_fp += fp
            total_fn += fn

        precision = total_tp / (total_tp + total_fp + 1e-3)
        recall = total_tp / (total_tp + total_fn + 1e-3)
        f1 = (2 * precision * recall) / (precision + recall + 1e-3)
        val_avg_loss = val_loss / len(val_loader)
        val_avg_iou = iou_loss / len(val_loader)
        train_epoch_loss = running / steps_per_epoch

        val_log = {
            "val/loss": val_avg_loss,
            "val/iou": val_avg_iou,
            "val/precision": precision,
            "val/recall": recall,
            "val/f1_score": f1,
            "val/learning_rate": current_lr
        }

        ckpt_path = os.path.join(ckpt_dir, f"best_epoch{epoch+1}_iou{val_avg_iou:.4f}.pth")
        torch.save({
            "epoch": epoch,
            "global_step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict() if 'scaler' in globals() else None,
            "best_val_iou": val_avg_iou,
        }, ckpt_path)
        print(
            f"epoch {epoch + 1}/{hyperparameter['epochs']} | "
            f"train_loss {train_epoch_loss:.4f} | "
            f"val_loss {val_avg_loss:.4f} | "
            f"val_iou {val_avg_iou:.4f} | "
            f"val_f1 {f1:.4f} | "
        )
        print()
        print(f"Saved new best checkpoint: {ckpt_path}")

    

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Using GPU: NVIDIA RTX A5000


  0%|          | 0/1152 [00:00<?, ?it/s]

epoch 8/10 | train_loss 0.1467 | val_loss 0.2616 | val_iou 0.0489 | val_f1 0.6070 | 

Saved new best checkpoint: checkpoints/v2/best_epoch8_iou0.0489.pth


  0%|          | 0/1152 [00:00<?, ?it/s]

W0513 13:04:25.043000 13079 torch/_inductor/utils.py:1731] [0/3] Not enough SMs to use max_autotune_gemm mode
W0513 13:06:12.454000 13079 torch/_functorch/_aot_autograd/autograd_cache.py:1101] [0/3] AOTAutograd cache unable to serialize compiled graph: Please convert all Tensors to FakeTensors first or instantiate FakeTensorMode with 'allow_non_fake_inputs'. Found in aten._to_copy.default(tensor([...], device='cuda:0', size=(4,), dtype=torch.uint8), device=device(type='cpu'))


epoch 9/10 | train_loss 0.2331 | val_loss 0.2906 | val_iou 0.0496 | val_f1 0.4880 | 

Saved new best checkpoint: checkpoints/v2/best_epoch9_iou0.0496.pth


  0%|          | 0/1152 [00:00<?, ?it/s]

epoch 10/10 | train_loss 0.1745 | val_loss 0.2352 | val_iou 0.0515 | val_f1 0.6094 | 

Saved new best checkpoint: checkpoints/v2/best_epoch10_iou0.0515.pth
